In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm

print(f"Target: Generate {MIT_DATASET_SIZE} samples with SINR levels: {SINR_LEVELS} dB")

In [ ]:
def load_all_h5(file_path):
    """Loads the entire dataset into memory for fast slicing."""
    with h5py.File(file_path, 'r') as f:
        data = f['dataset'][:] 
    return data

print("Loading Raw Sources...")
raw_clean = load_all_h5(MIT_CLEAN_FILE)
raw_noise = load_all_h5(MIT_NOISE_FILE)

print(f"Clean Source Shape: {raw_clean.shape}")
print(f"Noise Source Shape: {raw_noise.shape}")

# Validate shapes match expected MIT spec
# Clean frames are ~1.7ms, Noise frames are ~9.2ms
assert raw_clean.shape[1] >= MIT_SAMPLE_LENGTH, "Clean samples are too short!"
assert raw_noise.shape[1] >= MIT_SAMPLE_LENGTH, "Noise samples are too short!"

In [ ]:
def mix_signal(clean_signal, noise_signal, target_sinr_db):
    """
    Mixes a single clean signal with a single noise signal at a specific SINR.
    Physics: y = s + b * alpha
    """
    # Calculate the scalar for noise
    # SINR = P_signal / P_noise. 
    # Since inputs are Unit Power (P=1), P_noise_target = 1 / 10^(SINR/10)

    p_clean = np.mean(np.abs(clean_signal)**2)
    p_noise = np.mean(np.abs(noise_signal)**2)

    if p_noise == 0:
        return clean_signal, noise_signal

    # Linear ratio calculation
    sinr_linear = 10.0 ** (target_sinr_db / 10.0)
        
    # Scale factor (Amplitude is sqrt of Power)
    noise_scale = np.sqrt(p_clean / (p_noise * sinr_linear))
    
    # Mix
    # Scale the noise, not the signal.
    scaled_noise = noise_signal * noise_scale
    mixture = clean_signal + scaled_noise
    
    return mixture, scaled_noise

In [ ]:
def generate_dataset(n_samples, clean_source, noise_source, sinr_options):
    """
    Generates a dataset by randomly pairing clean and noise slices.
    """
    X_data = [] # Mixtures (Input)
    Y_data = [] # Clean Targets (Label)
    metadata = []
    
    n_clean_frames = clean_source.shape[0]
    n_noise_frames = noise_source.shape[0]
    
    print(f"Starting Generation: {n_samples} samples...")
    
    for i in tqdm(range(n_samples)):
        # Random Selection
        clean_idx = np.random.randint(0, n_clean_frames)
        noise_idx = np.random.randint(0, n_noise_frames)
        sinr = np.random.choice(sinr_options)
        
        # Slice Handling
        # We pick a random valid start point to increase variety
        
        # For Clean:
        clean_max_start = clean_source.shape[1] - MIT_SAMPLE_LENGTH
        c_start = np.random.randint(0, clean_max_start + 1) if clean_max_start > 0 else 0
        clean_slice = clean_source[clean_idx, c_start : c_start + MIT_SAMPLE_LENGTH]
        
        # For Noise:
        noise_max_start = noise_source.shape[1] - MIT_SAMPLE_LENGTH
        n_start = np.random.randint(0, noise_max_start + 1) if noise_max_start > 0 else 0
        noise_slice = noise_source[noise_idx, n_start : n_start + MIT_SAMPLE_LENGTH]
        
        mixture, _ = mix_signal(clean_slice, noise_slice, sinr)
        
        X_data.append(mixture)
        Y_data.append(clean_slice)
        
        # Metadata Logging
        meta_entry = {
            "global_index": i,
            "sinr_db": sinr,
            "clean_frame_id": clean_idx,
            "noise_frame_id": noise_idx,
            "interference_type": "spot (EMISignal1)", 
            "jamming_category": "Heavy" if sinr < -5 else "Light"
        }
        metadata.append(meta_entry)
        
    return np.array(X_data), np.array(Y_data), pd.DataFrame(metadata)

# --- RUN GENERATION ---
X_master, Y_master, meta_master = generate_dataset(MIT_DATASET_SIZE, raw_clean, raw_noise, SINR_LEVELS)

print(f"Generated Master Dataset: X={X_master.shape}, Y={Y_master.shape}")

In [ ]:
def save_master_dataset(save_path, X, Y, df):
    """
    Saves the full dataset as a single block (Tensors + CSV).
    """
    path = Path(save_path)
    path.mkdir(parents=True, exist_ok=True)
    
    # Define filenames
    
    # Save Tensors (Complex64 is standard for RF)
    np.save(MIT_SPOT_X, X.astype(np.complex64))
    np.save(MIT_SPOT_Y, Y.astype(np.complex64))
    
    # Save Metadata
    df.to_csv(MIT_SPOT_DATASET_METADATA_FILE, index=False)
    
    print(f"--- Save Complete ---")
    print(f"Location:  {path}")
    print(f"Files:     {MIT_SPOT_X.name}, {MIT_SPOT_Y.name}")
    print(f"Metadata:  {MIT_SPOT_DATASET_METADATA_FILE.name} ({len(df)} rows)")

save_master_dataset(MIT_GENERATED_DATA_DIR, X_master, Y_master, meta_master)